# LizyML Widget Tutorial — Binary Classification

This tutorial demonstrates using LizyML Widget for a **binary classification** task
with the [Breast Cancer Wisconsin](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) dataset.

You will learn:
1. Loading real-world data into the widget
2. Reviewing auto-detected settings
3. Configuring evaluation metrics (including `precision_at_k` with custom k)
4. Enabling calibration for probability estimates
5. Reviewing binary-specific metrics and plots (ROC curve, calibration, feature importance)
6. Hyperparameter tuning
7. Running inference with probability predictions

## 0. Installation

```bash
pip install lizyml-widget lizyml[plots,tuning,calibration,explain]
```

## 1. Load the Breast Cancer Dataset

This dataset contains 569 samples with 30 features computed from
digitized images of fine needle aspirates of breast masses.
The target is binary: malignant (0) or benign (1).

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")
print(f"  0 = malignant, 1 = benign")
df.head()

## 2. Launch the Widget

In [ ]:
from lizyml_widget import LizyWidget

w = LizyWidget()
w.load(df, target="target")
w

## 3. Data Tab — Auto-Detection

The widget auto-detects:
- **Task**: `binary` (2 unique target values)
- **CV**: `stratified_kfold` (preserves class balance)
- **Columns**: All 30 features included

In [ ]:
print(f"Task:     {w.task}")
print(f"CV:       {w.cv_method} ({w.cv_n_splits} folds)")
print(f"Shape:    {w.df_shape}")

## 4. Config — Model & Evaluation Settings

For binary classification, we configure:
- Multiple evaluation metrics including `precision_at_k` with custom k
- Calibration enabled (Platt scaling) for well-calibrated probabilities

In [ ]:
w.set_config({
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 4,
        },
    },
    "training": {
        "seed": 42,
        "early_stopping": {"enabled": True, "rounds": 50},
    },
    "evaluation": {
        "metrics": ["auc", "logloss", "f1", "precision_at_k"],
        "params": {"precision_at_k_k": 20},
    },
    "calibration": {
        "method": "platt",
        "n_splits": 5,
        "params": {},
    },
})

print("Config set with AUC, LogLoss, F1, precision_at_k (k=20), and Platt calibration")

## 5. Run Fit

In [ ]:
w.fit()

## 6. Results — Binary Classification Metrics & Plots

After fitting, the Results tab shows:

**Metrics** (IS = in-sample, OOS = out-of-sample):
- **AUC** — Area Under ROC Curve
- **LogLoss** — Logarithmic Loss
- **F1** — F1 Score
- **precision_at_k (k=20)** — Precision among top-k predictions

**Binary-specific plots**:
- **ROC Curve** — Receiver Operating Characteristic
- **Calibration** — Predicted vs actual probability (when calibration enabled)
- **Probability Histogram** — Distribution of predicted probabilities
- **Feature Importance** — Split, Gain, and SHAP variants

In [ ]:
summary = w.get_fit_summary()
if summary:
    print(f"Fold count: {summary.fold_count}")
    for name, vals in summary.metrics.items():
        if isinstance(vals, dict):
            oos = vals.get("oos", "N/A")
            print(f"  {name}: OOS={oos}")

## 7. Tune — Hyperparameter Optimization

In [ ]:
w.tune()

tune_summary = w.get_tune_summary()
if tune_summary:
    print(f"Best score:  {tune_summary.best_score:.4f}")
    print(f"Metric:      {tune_summary.metric_name} ({tune_summary.direction})")
    print(f"Best params: {tune_summary.best_params}")
    print(f"Trials:      {len(tune_summary.trials)}")

## 8. Inference — Predict with Probabilities

For binary classification, predictions include probability estimates.

In [ ]:
df_test = df.drop(columns=["target"]).tail(10)

result = w.predict(df_test)
if result:
    print(f"Predictions shape: {result.predictions.shape}")
    print(f"Columns: {list(result.predictions.columns)}")
    result.predictions.head()

## 9. Save Config

In [ ]:
w.save_config("binary_config.yaml")
print("Config saved to binary_config.yaml")

## 10. Export Code

In [ ]:
# Export standalone training code (no LizyML dependency needed)
export_path = w.export_code("./exported_code")
print(f"Code exported to: {export_path}")
print("Generated files:")
for f in sorted(export_path.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(export_path)}")